# 32 — Setup selection, review checks and the analysis brief

Thin notebook: it only **imports** and **calls** `src/setup_analysis.py`, which runs each step's module in `src/`, and **displays** a few rows of what each step wrote.

**What it reproduces:** everything the DINOv2 analysis document rests on beyond notebooks 30 and 31: the registers check, the held-out (split-half) choice of model, layer and token, the drawing and photo preparation checks and their controls, the view erasure, the review round of 2026-09-30 item by item, the registers twin reports, the patent 3-D pages, the attention decision figure, the directory classes across the development years and their maps window by window (brief 4.5), the short analysis document, the default output since 2026-10-01 (Markdown, PDF, Word; step 12), and optionally the 30-page brief (step 11).

**Input** (written upstream by `eVTOL-Embedding-Extraction`, notebooks 20 to 25, not run here): the embeddings of `dinov2-large_518` and `dinov2-reg-large_518` for both sources, the preprocessing variants and controls (`preprocess_variants/{thick1, thick2, grey, contrast}`), the crop embeddings, the CLS attention arrays and `attention/registers_share.csv`; the evtol.news development years (`EVTOLNEWS_DS/0_source/years/aircraft_dev_year.csv`).

**Output:** `1639_LABELLED/3_embedding_evaluation/{registers_check, setup_selection, view_erasure, review_fixes, umap_3d, class_evolution}/`, the twin reports, and `docs/embedding_evaluation/analysis_document/ANALYSIS_BRIEF.md` with its Word and PDF copies in `Embedding Analysis brief/current/`.

**Idempotent.** A step runs only when one of its outputs is missing or older than its newest input (inputs are data files, not code); otherwise it prints `skipped` and why. `FORCE = True` reruns every step; `FORCE_STEPS` reruns only the named ones (use it after changing a step's code). `RUN_TWINS` switches the slow twin reports on. `BRIEF_REUSE` rebuilds the brief's text without redoing its two slow computations (the class-mix draws and the exp4 subset), which the last full build cached.

The first cell hashes the key result files; the last cell checks them again.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'config.yaml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display
from src import setup_analysis as SA

FORCE = False            # rerun every step
FORCE_STEPS = set()      # e.g. {'analysis_brief'} after a code change in that step
RUN_TWINS = False        # the registers twin reports take about half an hour
RUN_FULL_BRIEF = False   # the 30-page brief (full_version/); the short document is the default
BRIEF_REUSE = True       # reuse the cached class-mix draws and exp4 subset of the last full brief build


def forced(name):
    return FORCE or name in FORCE_STEPS


print(sys.executable)
before = SA.snapshot(SA.KEY_FILES)
print(len(before), 'key files hashed')

/home/vasco/anaconda3/bin/python
17 key files hashed


## 1. Registers check

`src/registers_check.py`: kNN-5 balanced accuracy of every matrix under both models, on the primary image rule of each source (drawings: exp3 main figure, 12 G1 types and 5 parent classes; photos: hero image, 5 classes), with a paired bootstrap of registers minus base at layer 24 CLS. Writes `registers_check/registers_knn.csv` and `registers_paired.csv`.

In [2]:
SA.run('registers_check', forced('registers_check'))
pd.read_csv(SA.REGCHK / 'registers_paired.csv')

[registers_check] skipped: outputs newer than inputs


,source,label,matrix,diff,ci_lo,ci_hi,agree
0,patents,G1 12-class,L24 CLS,0.056511,0.007978,0.102693,0.447964
1,patents,parent 5-class,L24 CLS,0.111425,0.025818,0.198657,0.696833
2,photos,5-class,L24 CLS,0.029684,-0.001288,0.062925,0.675022


## 2. Held-out selection of model, layer and token (S1a)

`src/heldout_selection.py` (`run_s1a`): the split-half rule over makers (200 splits × 2 halves, seed 42) on 12 candidates (2 models × 3 layers × 2 tokens) and on the 6 registers candidates alone; selection frequency, held-out score, optimism gap, and the decision. Writes `setup_selection/s1a_*.csv`.

In [3]:
SA.run('heldout_selection', forced('heldout_selection'))
pd.read_csv(SA.SEL / 's1a_decision.csv')

[heldout_selection] skipped: outputs newer than inputs


,candidate,selection_freq,fixed_heldout_mean,chosen
0,registers L24 CLS,0.687500,0.402936,True
1,registers L22 CLS,0.285000,0.378125,False
2,registers L24 patch mean,0.024375,0.279674,False
3,registers L18 CLS,0.001875,0.245348,False
4,registers L22 patch mean,0.001250,0.248735,False
5,registers L18 patch mean,0.000000,0.232331,False


## 3. Preprocessing variants (S1b, S2, S3, fairness)

`src/preprocess_variants_eval.py`: thick1 and thick2 drawings against plain, greyscale photos against colour, the split-half choice of each source's preparation, and photos against drawings on the shared 5 classes. Writes `setup_selection/s2s3_*.csv`, `s1b_*.csv`, `fairness*.csv` and `SETUP_SELECTION.md`.

In [4]:
SA.run('preprocess_variants', forced('preprocess_variants'))
pd.read_csv(SA.SEL / 's2s3_paired.csv')

[preprocess_variants] skipped: outputs newer than inputs


,source,label,matrix,comparison,diff,ci_lo,ci_hi,agree
0,patents,patents G1 12-class,L24 CLS,thick1 - plain,-0.033978,-0.075984,0.005300,0.496229
1,patents,patents G1 12-class,L24 CLS,thick2 - plain,-0.050603,-0.098014,-0.005038,0.410256
2,patents,patents parent 5-class,L24 CLS,thick1 - plain,-0.042915,-0.103384,0.023784,0.708899
3,patents,patents parent 5-class,L24 CLS,thick2 - plain,-0.099021,-0.189163,-0.013081,0.663650
4,photos,photos 5-class,L24 CLS,grey - colour,0.006318,-0.012331,0.025668,0.853068


## 4. Preprocessing controls (grey, contrast)

`src/preprocess_controls_eval.py`: the two controls that separate colour, darkness and width in the thick variants, the split-half choice over the five drawing preparations, and grey drawings against grey photos. Writes `setup_selection/s2_controls_*.csv`, `s1b_v2_*.csv`, `fairness_grey_grey.csv` and `SETUP_SELECTION_addendum.md`.

In [5]:
SA.run('preprocess_controls', forced('preprocess_controls'))
pd.read_csv(SA.SEL / 's1b_v2_decision.csv')

[preprocess_controls] skipped: outputs newer than inputs


,candidate,selection_freq,fixed_heldout_mean,source,chosen
0,plain,0.37375,0.261803,patents,True
1,thick1,0.24750,0.247849,patents,False
2,grey,0.23125,0.261339,patents,False
3,contrast,0.11875,0.231066,patents,False
4,thick2,0.02875,0.211302,patents,False


## 5. View erasure

`src/view_erasure_check.py`: for both models at layer 24 CLS, the linear erasure of the view fitted on the 1 541 whole-aircraft figures, the architecture scores before and after, a random and a variance-matched random removal, and the view probe with the erasure fitted on half of the makers. Writes `view_erasure/view_erasure.csv`.

In [6]:
SA.run('view_erasure', forced('view_erasure'))
ve = pd.read_csv(SA.VEDIR / 'view_erasure.csv')
ve[ve.measure == 'paired_diff'][['tag', 'section', 'variant', 'value', 'ci_lo', 'ci_hi']]

[view_erasure] skipped: outputs newer than inputs


,tag,section,variant,value,ci_lo,ci_hi
50,dinov2-large_518,"architecture kNN-5, G1 12-class",view erasure - original,0.004510,-0.021614,0.031711
51,dinov2-large_518,"architecture kNN-5, G1 12-class",random directions - original,0.002626,-0.000865,0.007759
52,dinov2-large_518,"architecture kNN-5, G1 12-class",variance-matched random - original,-0.014610,-0.049951,0.024343
57,dinov2-large_518,"architecture kNN-5, G1 12-class, Perspective m...",view erasure - original,-0.003114,-0.023355,0.016016
58,dinov2-large_518,"architecture kNN-5, G1 12-class, Perspective m...",random directions - original,0.001634,0.000000,0.005319
59,dinov2-large_518,"architecture kNN-5, G1 12-class, Perspective m...",variance-matched random - original,-0.048396,-0.110869,-0.011787
64,dinov2-large_518,"architecture kNN-5, parent 5-class",view erasure - original,0.009831,-0.017365,0.034004
65,dinov2-large_518,"architecture kNN-5, parent 5-class",random directions - original,-0.001561,-0.004285,0.000000
66,dinov2-large_518,"architecture kNN-5, parent 5-class",variance-matched random - original,-0.040878,-0.071096,-0.011608
71,dinov2-large_518,"architecture kNN-5, parent 5-class, Perspectiv...",view erasure - original,0.041657,0.015542,0.077767


## 6. Review fixes (review round of 2026-09-30)

`src/review_fixes.py` (`run_items`), one item at a time: M3 pool size and class mix, M4 held-out erasure and the 200-draw control, M6/M9 per-class recall with Wilson intervals and the size-balanced reading, M11 count reconciliations, S2 probe intervals and shuffled chance, S3 k sensitivity and image rules, S4 every paired contrast with maker intervals and Holm, S6 matching intervals, S7/S8 split-half details and the joint grid, S9 attention extras, S10 audit and κ, S11 the view ratio and the figure UMAP. Then the seven review figures and `INDEX.md`. Writes `review_fixes/`.

In [7]:
rf = SA.run_review_fixes(forced('review_fixes'))
pd.Series(rf).value_counts()

[review_fixes:m3] skipped: outputs newer than inputs
[review_fixes:m4] skipped: outputs newer than inputs
[review_fixes:m6] skipped: outputs newer than inputs
[review_fixes:m11] skipped: outputs newer than inputs
[review_fixes:s2_12] skipped: outputs newer than inputs
[review_fixes:s2_5] skipped: outputs newer than inputs
[review_fixes:s2_ph] skipped: outputs newer than inputs
[review_fixes:s3] skipped: outputs newer than inputs
[review_fixes:s4] skipped: outputs newer than inputs
[review_fixes:s6] skipped: outputs newer than inputs
[review_fixes:s78] skipped: outputs newer than inputs
[review_fixes:s9] skipped: outputs newer than inputs
[review_fixes:s10] skipped: outputs newer than inputs
[review_fixes:s11] skipped: outputs newer than inputs
[review_fixes:s11_umap] skipped: outputs newer than inputs
[review_fixes:figures] running: input newer than output (S4_paired_contrasts.csv)


[review_fixes] figure /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/3_embedding_evaluation/review_fixes/figs/FA_forest_paired_contrasts.png


[review_fixes] figure /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/3_embedding_evaluation/review_fixes/figs/FB_recall_vs_pool_share.png


[review_fixes] figure /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/3_embedding_evaluation/review_fixes/figs/FC_optimism_dumbbell.png


[review_fixes] figure /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/3_embedding_evaluation/review_fixes/figs/FD_class_counts.png


...

[review_fixes] figure /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/3_embedding_evaluation/review_fixes/figs/FF_view_erasure_control.png


[review_fixes] figure /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/3_embedding_evaluation/review_fixes/figs/FG_umap_figures_by_view.png


[review_fixes:index] running: input newer than output (S4_paired_contrasts.csv)


[review_fixes] index /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/3_embedding_evaluation/review_fixes/INDEX.md


skipped    15
Name: count, dtype: int64

## 7. Registers twin reports (optional, slow)

`src/registers_reports.py`: the two embedding reports of notebooks 30 and 31 rebuilt on `dinov2-reg-large_518` (`PATENT_EMBEDDING_ANALYSIS_REGISTERS.md`, `EVTOLNEWS_EMBEDDING_ANALYSIS_REGISTERS.md`, their tables and the registers matching table). The brief reads their tables. Runs only with `RUN_TWINS = True`.

In [8]:
SA.run_twins(RUN_TWINS, forced('registers_reports'))

[registers_reports] not enabled (RUN_TWINS = False)


'disabled'

## 8. Patent 3-D pages (base and registers)

`src/patent_3d.py`: the rotatable 3-D UMAP page of the patent embeddings for each model. Writes `umap_3d/PATENT_UMAP_3D.html` and `PATENT_UMAP_3D_REGISTERS.html`.

In [9]:
SA.run('patent_3d', forced('patent_3d'))
SA.run('patent_3d_registers', forced('patent_3d_registers'))

[patent_3d] skipped: outputs newer than inputs
[patent_3d_registers] skipped: outputs newer than inputs


'skipped'

## 9. Attention figure

`src/registers_attention_figure.py`: base against registers CLS attention at layers 18, 22 and 24 on the same drawings and photos, from the attention arrays of the extraction repo. Writes `registers_check/attention_base_vs_registers.png`. (The brief draws its own attention figures in step 11.)

In [10]:
SA.run('attention_figure', forced('attention_figure'))

[attention_figure] skipped: outputs newer than inputs


'skipped'

## 10. The directory classes across the development years (brief 4.5)

`src/class_evolution.py`: the evtol.news hero images (registers, layer 24 CLS, maker held out) dated by the development year both language-model readers agreed on (`EVTOLNEWS_DS/0_source/years/aircraft_dev_year.csv`, status `agreed` only; not yet reviewed by the author), in the Labelling Analysis windows. (1) Spearman ρ between each aircraft's distance to its class centroid (other makers only) and the year, per class and pooled, maker bootstrap; (2) the 2020-26 aircraft classified by kNN-5 from neighbours of ≤ 2019 against neighbours of 2020-26, pools matched class by class (200 draws); (3) the move of each class centre from 2016-19 to 2020-23 against a permutation band (999, seed 42); (4) tests 1 and 3 on the drawings' parent classes by priority year. Writes `class_evolution/ce_*.csv` (numbers only).

In [11]:
SA.run('class_evolution', forced('class_evolution'))
pd.read_csv(SA.CEDIR / 'ce_drift.csv')

[class_evolution] running: input newer than output (aircraft_identity_ALL.xlsx)


wrote ce_coverage.csv, ce_counts.csv, ce_convergence.csv, ce_median_distance.csv, ce_temporal_knn.csv, ce_drift.csv, ce_shares.csv to /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_LABELLED/3_embedding_evaluation/class_evolution


,source,class,n_2016_19,n_2020_23,tested,distance,perm_median,perm_p95,excess,p,reading,p_holm
0,photos,VT,94,150,True,0.024738,0.010853,0.017200,0.013884,0.003,moved (above the permutation band),0.021
1,photos,LC,56,81,True,0.014193,0.014865,0.023774,-0.000672,0.578,no drift detected,0.670
2,photos,WM,87,116,True,0.013869,0.011050,0.017091,0.002819,0.161,no drift detected,0.645
3,photos,ER,18,34,True,0.027644,0.021040,0.033214,0.006604,0.151,no drift detected,0.645
4,photos,HB,52,21,True,0.056006,0.051625,0.075122,0.004381,0.335,no drift detected,0.670
5,drawings,VT,130,142,True,0.018322,0.006312,0.010413,0.012010,0.001,moved (above the permutation band),0.008
6,drawings,LC,54,82,True,0.018819,0.013445,0.022162,0.005374,0.129,no drift detected,0.645
7,drawings,WM,17,16,True,0.101729,0.063179,0.097443,0.038550,0.037,moved (above the permutation band),0.222
8,drawings,ER,4,1,False,NaN,NaN,NaN,NaN,NaN,not tested (fewer than 15 aircraft in a window),NaN
9,drawings,HB,15,5,False,NaN,NaN,NaN,NaN,NaN,not tested (fewer than 15 aircraft in a window),NaN


## 10b. The maps window by window (brief 4.5)

`src/umap_by_window.py`: ONE UMAP fit per source on all its aircraft (the document's own 2-D maps: 663 drawings; all 1 157 photo aircraft, the undated ones in the fit but in no window), never a fit per window, so positions compare across windows. Years and dated sets from `class_evolution`'s loaders (949 dated photo aircraft, 663 drawings). Per window: each class's centre (mean map position) and n; the class separation ratio within each window with a joint maker bootstrap and the difference 2020-23 minus 2016-19. Writes `class_evolution/umap_by_window_*.csv`, `umap_3d/PATENT_UMAP_3D_BY_WINDOW.html` and `EVTOLNEWS_DS/3_embedding_evaluation/umap_3d/EVTOLNEWS_UMAP_3D_BY_WINDOW.html` (private; 3-D fit, plotly frames with a window slider). The two 2-D figures are drawn by the brief step.

In [12]:
SA.run('umap_by_window', forced('umap_by_window'))
pd.read_csv(SA.CEDIR / 'umap_by_window_separation.csv')

[umap_by_window] running: input newer than output (aircraft_identity_ALL.xlsx)


[drawings] 663 aircraft in the fit, 663 dated; ≤ 2011 69, 2012-15 67, 2016-19 220, 2020-23 246, 2024-26 (partial) 61


[photos] 1157 aircraft in the fit, 949 dated; ≤ 2011 44, 2012-15 65, 2016-19 307, 2020-23 402, 2024-26 (partial) 131


wrote umap_by_window_coords_drawings.csv, umap_by_window_coords_photos.csv, umap_by_window_centroids.csv, umap_by_window_separation.csv, PATENT_UMAP_3D_BY_WINDOW.html, EVTOLNEWS_UMAP_3D_BY_WINDOW.html


,source,window,n_aircraft,n_makers,ratio,ci_lo,ci_hi
0,drawings,<= 2011,69.0,48.0,1.092084,1.017646,1.202322
1,drawings,2012-15,67.0,46.0,1.095982,1.009929,1.235099
2,drawings,2016-19,220.0,119.0,1.088857,1.051717,1.129627
3,drawings,2020-23,246.0,149.0,1.084589,1.040898,1.143726
4,drawings,2024-26 (partial),61.0,50.0,1.075290,1.008658,1.156737
5,drawings,2020-23 minus 2016-19,NaN,NaN,-0.004268,-0.063566,0.062545
6,photos,<= 2011,44.0,27.0,1.100148,1.009870,1.345414
7,photos,2012-15,65.0,43.0,1.122786,1.044320,1.225527
8,photos,2016-19,307.0,201.0,1.102843,1.068237,1.143183
9,photos,2020-23,402.0,228.0,1.121511,1.088895,1.160247


## 11. The 30-page analysis brief (optional)

`src/analysis_brief.py` writes `ANALYSIS_BRIEF.md` and its figures, every number read from the tables above; `src/brief_build.py` prints the PDF and the Word file into `Embedding Analysis brief/full_version/` (name ending in "full"). Superseded on 2026-10-01 by the short document of step 12; it runs only with `RUN_FULL_BRIEF = True`.

In [13]:
if RUN_FULL_BRIEF:
    SA.run_brief(forced('analysis_brief'), reuse=BRIEF_REUSE)
    print(SA.BRIEF_MD.read_text(encoding='utf-8').count('\n'), 'lines in', SA.BRIEF_MD.name)
else:
    print('[analysis_brief] not enabled (RUN_FULL_BRIEF = False)')

[analysis_brief] not enabled (RUN_FULL_BRIEF = False)


## 12. The short analysis document (default output)

`src/analysis_simple.py` (2026-10-01, after the author's 42 comments on the 30-page brief): the pipeline on one page with the index of the tests, a glossary with a worked example of the 5-neighbour vote, the data and the evtol.news steps (who did each, what is left to review), the few settings that change the result, drawings, photos, drawings against photos, change over time (with links and QR codes to the 3-D maps), conclusions with one robustness table. Every number is read from the tables above; the one new check (the vote with the same-company rule switched off) is written once to `review_fixes/N1_same_company_rule.csv`. Output: `ANALYSIS_SIMPLE.md` and `Embedding Analysis brief/current/Embedding Analysis — Visual Learning (private).docx` / `.pdf`; the Word step reports the LibreOffice page count and fails above 20 pages.

In [14]:
SA.run_simple(forced('analysis_simple'))
print(SA.SIMPLE_MD.read_text(encoding='utf-8').count('\n'), 'lines in', SA.SIMPLE_MD.name)

[analysis_simple] running: input newer than output (umap_by_window_separation.csv)


wrote /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/eVTOL-Visual-Evaluation/archive/docs/embedding_evaluation/analysis_document/ANALYSIS_SIMPLE.md
  Figure 1: /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/eVTOL-Visual-Evaluation/archive/docs/embedding_evaluation/analysis_document/figs_simple/s_pipeline.png
  Figure 2: /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/eVTOL-Visual-Evaluation/archive/docs/embedding_evaluation/analysis_document/figs_simple/s_knn_example.png
  Figure 3: /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/eVTOL-Visual-Evaluation/archive/docs/embedding_evaluation/analysis_document/figs_brief/f_examples_patents.png
  Figure 4: /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/EVTOLNEWS_DS/3_embedding_evaluation/analysis_brief_figs/f02_examples.png
  Figure 5: /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/EVTOLNEWS_DS/3_embedding_evaluation/analysis_brief_figs/f_attention_base_vs_registers.png
  Figure 6: /home

/mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/Embedding Analysis brief/current/Embedding Analysis — Visual Learning (private).pdf 4396394
wrote /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/Embedding Analysis brief/current/Embedding Analysis — Visual Learning (private).pdf  (10 pages)


source  /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/eVTOL-Visual-Evaluation/archive/docs/embedding_evaluation/analysis_document/._clean_k_882jjy.md
        11 image references, title 'Visual Learning: DINOv2 embeddings of patent drawings and ph'
boxes   4 answer, 0 methods
images  11 distinct embedded (expected 11)
stray   0 occurrences of '[image:'
wrote   /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/Embedding Analysis brief/current/._clean_export.docx  (3,946,002 bytes)
 /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/eVTOL-Visual-Evaluation/archive/labeling_evaluation/scripts/export_brief_docx.py:427: UserWarning: style lookup by style_id is deprecated. Use style name as key instead.
  st = doc.styles[name]

clean: /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/Embedding Analysis brief/current/Embedding Analysis — Visual Learning (private).docx  (em dashes left in text: 0)


wrote /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/Embedding Analysis brief/current/Embedding Analysis — Visual Learning (private).docx  (LibreOffice: 10 pages, cap 20)
357 lines in ANALYSIS_SIMPLE.md


## Check: are the key results unchanged?

With `FORCE = False` and nothing upstream changed, every key file must hash as before.

In [15]:
after = SA.snapshot(SA.KEY_FILES)
check = pd.DataFrame([{'file': Path(k).name, 'unchanged': before.get(k) == v} for k, v in after.items()])
display(check)
assert check.unchanged.all() or FORCE or FORCE_STEPS, 'a key result changed'

,file,unchanged
0,registers_knn.csv,True
1,registers_paired.csv,True
2,s1a_candidates.csv,True
3,s1a_decision.csv,True
4,s2s3_knn.csv,True
5,s2_controls_knn.csv,True
6,s1b_v2_decision.csv,True
7,view_erasure.csv,True
8,S4_paired_contrasts.csv,True
9,M3_pool_size_and_mix.csv,True
